In [ ]:
import xarray as xr
import os

In [ ]:
start_dir = os.getcwd()
watershed = "watersheds/huc8_07120004/"
resolution = 30

In [ ]:
event_indices_list = list(range(120))
event_indices_list.remove(7)
event_indices_list.remove(17)
event_indices_list.remove(44)
event_indices_list.remove(60)
event_indices_list.remove(66)
event_indices_list.remove(67)
event_indices_list.remove(72)

In [ ]:
shape_dict = {
    '(39, 13)': 0, 
    '(39, 14)': 0, 
    '(40, 13)': 0,
    '(40, 14)': 0
}

In [ ]:
for event_idx in event_indices_list:
    file_path = os.path.join(
        start_dir, watershed, f'data_for_HiPIMS_{resolution}',
        'precipitation_st4_midwest_fill_nan', 
        f"event_{event_idx}.nc"
    )

    adjust_file_path = os.path.join(
        start_dir, watershed, f'data_for_HiPIMS_{resolution}',
        'precipitation_st4_midwest_fill_nan_adjust', 
        f"event_{event_idx}.nc"
    ) 

    pr_ds = xr.open_dataset(file_path)
    pr = pr_ds["prcp"]

    # shape corrections
    if (pr.shape[1], pr.shape[2]) == (39, 13):
        shape_dict['(39, 13)'] += 1

    elif (pr.shape[1], pr.shape[2]) == (39, 14):
        shape_dict['(39, 14)'] += 1
        # pr = pr[:, :39, :13]

    elif (pr.shape[1], pr.shape[2]) == (40, 13):
        shape_dict['(40, 13)'] += 1
        # pr = pr[:, :39, :13]

    elif (pr.shape[1], pr.shape[2]) == (40, 14):
        shape_dict['(40, 14)'] += 1
        # pr = pr[:, :39, :13]

    else:
        print('New shape detected', pr.shape)
        continue


    # Slice pr to the target shape
    pr_fixed = pr[:, :39, :13].copy()

    # Build coordinate dict with correct slicing
    coords_fixed = {}
    for k, v in pr_ds.coords.items():
        if v.dims == ("x",):
            coords_fixed[k] = v[:13]
        elif v.dims == ("y",):
            coords_fixed[k] = v[:39]
        else:
            coords_fixed[k] = v

    # Build the new dataset
    fixed_ds = xr.Dataset({"prcp": pr_fixed}, coords=coords_fixed)

    # Save back, overwrite
    fixed_ds.to_netcdf(adjust_file_path, mode='w')


In [ ]:
shape_dict = {
    '(39, 13)': 0, 
    '(39, 14)': 0, 
    '(40, 13)': 0,
    '(40, 14)': 0
}

In [ ]:
for event_idx in event_indices_list:
    pr_ds = xr.open_dataset(os.path.join(start_dir, watershed, f'data_for_HiPIMS_{resolution}','precipitation_st4_midwest_fill_nan_adjust', f"event_{event_idx}.nc"))  # st4
    pr = pr_ds["prcp"]
    if (pr.shape[1], pr.shape[2]) == (39, 13):
        shape_dict['(39, 13)'] += 1
    elif (pr.shape[1], pr.shape[2]) == (39, 14):
        shape_dict['(39, 14)'] += 1        
    elif (pr.shape[1], pr.shape[2]) == (40, 13):
        shape_dict['(40, 13)'] += 1
    elif (pr.shape[1], pr.shape[2]) == (40, 14):
        shape_dict['(40, 14)'] += 1
    else:
        print('New shape detected')